# Snowflake S3 Integration

**AWS sandbox + Snowflake trial: access-key lab.**

Use one orders dataset throughout. Connect a private S3 bucket to a named external stage, list files, inspect directory metadata, query CSV rows, and load `ORDERS`. An optional section queries the same files through `ExternalOrderTable`.

This is an instructional notebook: copy each SQL block into a **Snowsight SQL worksheet**, JSON into the AWS IAM policy editor, and optional shell commands into an AWS CLI terminal. It does not require a Python connection or notebook SQL magic. Run sections in order, choosing only one credential option.

The examples use the database and schema selected at the top of your Snowsight worksheet. Stage and table names are unqualified so they use that selected context. Replace `YOUR_UNIQUE_BUCKET` and credential placeholders only. No real credentials are included. Keep this shared notebook free of real keys; enter them only in your private worksheet and remove them before sharing/exporting it.

Access keys work with Snowflake directly; a `STORAGE INTEGRATION` object and cross-account role are not prerequisites for this route. [Snowflake IAM-user authentication](https://docs.snowflake.com/en/user-guide/data-load-s3-config-aws-iam-user)

## 1. Choose the easiest available credential route

| What the sandbox gives you | Use |
|---|---|
| Access key ID + secret access key + session token | Temporary-credential stage in section 6B; all three values are required |
| IAM-user access key ID + secret access key | Two-key stage in section 6A |
| Console login only, with permission to create a lab IAM user/key | Create the small read-only user in section 4 |
| Console login only, with IAM key creation denied | This route is blocked; see section 13 |

An AWS console password is not an API secret key. Do not assume the sandbox exposes API credentials: inspect its connection details first.

At the top of your Snowsight worksheet, select an existing warehouse, database, schema, and a role permitted to create stages, file formats, and tables there. Keep that selection throughout the exercise. Corporate accounts may impose additional restrictions.

## 2. Create the bucket in the AWS sandbox

1. Start the AWS sandbox and open its AWS console.
2. Open **S3 ? Create bucket**. Choose a general-purpose bucket in a region allowed by the sandbox.
3. Enter a globally unique name, for example `orders-yourname-uniqueid`. Substitute your actual name for `YOUR_UNIQUE_BUCKET` throughout this notebook.
4. Keep **Block all public access** enabled and ACLs disabled. Use default S3-managed encryption (SSE-S3) for this lab if the sandbox permits it.
5. Open the bucket and create a folder named `orders`.

The stage will point to `s3://YOUR_UNIQUE_BUCKET/orders/`. Use a trailing slash. The bucket belongs to the AWS sandbox; it is not Snowflake-managed storage. Prefer the Snowflake region when available, but use the sandbox's permitted region.

An existing bucket forced to use KMS may need additional key permissions; selecting SSE-S3 for a new lab bucket avoids that extra setup.

## 3. Upload the first orders file

The companion file `orders_01.csv` is provided beside this notebook:

```csv
order_id,order_date,customer_name,status,order_total
1001,2026-09-10,Asha,NEW,120.50
1002,2026-09-10,Ravi,SHIPPED,250.00
1003,2026-09-11,Meena,NEW,75.25
```

In **AWS S3 ? your bucket ? orders/ ? Upload ? Add files**, select `orders_01.csv` and upload it. Confirm its object key is `orders/orders_01.csv`. Upload only the first file now.

For this external stage, upload through AWS S3, not the Snowsight internal-stage upload screen. Snowflake `PUT` targets internal stages.

Optional alternative, in a terminal with AWS CLI already authenticated to the sandbox:

```bash
aws sts get-caller-identity
aws s3 cp orders_01.csv s3://YOUR_UNIQUE_BUCKET/orders/orders_01.csv
aws s3 ls s3://YOUR_UNIQUE_BUCKET/orders/
```

`get-caller-identity` identifies the CLI principal; verify it is the intended sandbox identity. The CLI identity may differ from the credentials you later give Snowflake.

## 4. Only if needed: create a read-only IAM user and access key

Skip this section if you already have permitted API credentials with the required S3 access. No trust relationship or cross-account role is involved here.

In **IAM ? Policies ? Create policy ? JSON**, paste the policy below after replacing the bucket name. Name it `OrdersReadPolicy`. It permits reading only the orders objects, listing the orders prefix, and obtaining the bucket location.

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "BucketLocation",
      "Effect": "Allow",
      "Action": "s3:GetBucketLocation",
      "Resource": "arn:aws:s3:::YOUR_UNIQUE_BUCKET"
    },
    {
      "Sid": "ListOrders",
      "Effect": "Allow",
      "Action": "s3:ListBucket",
      "Resource": "arn:aws:s3:::YOUR_UNIQUE_BUCKET",
      "Condition": {"StringLike": {"s3:prefix": ["orders/", "orders/*"]}}
    },
    {
      "Sid": "ReadOrders",
      "Effect": "Allow",
      "Action": ["s3:GetObject", "s3:GetObjectVersion"],
      "Resource": "arn:aws:s3:::YOUR_UNIQUE_BUCKET/orders/*"
    }
  ]
}
```

In **IAM ? Users ? Create user**, create `snowflake-orders-reader` without console access and attach `OrdersReadPolicy`. Open that user's **Security credentials ? Access keys ? Create access key**. Select the applicable external-application use case and record the key ID and secret privately. The secret is shown only during creation.

Use this user's keys in section 6A. Upload with your original sandbox console identity: this read-only user cannot upload or delete objects. Do not create root access keys or modify sandbox-managed users/roles. If IAM denies creation, proceed to the blocked-route guidance rather than changing sandbox controls.

Sources: [required S3 permissions](https://docs.snowflake.com/en/user-guide/data-load-s3-config-aws-iam-user), [AWS access-key management](https://docs.aws.amazon.com/IAM/latest/UserGuide/access-key-self-managed.html).

## 5. Use the selected Snowsight context

At the top of the Snowsight SQL worksheet, select your existing **database, schema, warehouse, and role**. All objects below are created in that selected schema. No database, schema, or warehouse creation is needed.

Run this read-only check to confirm your selection:

```sql
SELECT CURRENT_DATABASE() AS SELECTED_DATABASE,
       CURRENT_SCHEMA() AS SELECTED_SCHEMA,
       CURRENT_WAREHOUSE() AS SELECTED_WAREHOUSE,
       CURRENT_ROLE() AS SELECTED_ROLE;
```

If a database, schema, or warehouse is missing, select it in the worksheet before continuing. Then create the CSV file format:

```sql
CREATE OR REPLACE FILE FORMAT ORDERS_CSV
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  EMPTY_FIELD_AS_NULL = TRUE
  DATE_FORMAT = 'YYYY-MM-DD';
```

Keep the same worksheet context for the following SQL, including cleanup. `SKIP_HEADER=1` skips the header in each input file. The optional external table is named `ExternalOrderTable`; Snowflake displays unquoted identifiers in uppercase.

## 6A. Create the stage with IAM-user access keys

**Run 6A or 6B, not both.** This option is for a key pair that does not have a session token.

```sql
CREATE STAGE ORDERS_S3_STAGE
  URL = 's3://YOUR_UNIQUE_BUCKET/orders/'
  CREDENTIALS = (
    AWS_KEY_ID = 'REPLACE_WITH_ACCESS_KEY_ID'
    AWS_SECRET_KEY = 'REPLACE_WITH_SECRET_ACCESS_KEY'
  )
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = FALSE)
  FILE_FORMAT = (FORMAT_NAME = ORDERS_CSV);

LIST @ORDERS_S3_STAGE;
```

Expected: a listing containing `orders_01.csv`. Stage creation alone is not proof of readable data; continue through the row query in section 8. If the stage already exists, inspect it and update credentials with section 12 instead of replacing a stage referenced by an external table.

[CREATE STAGE syntax](https://docs.snowflake.com/en/sql-reference/sql/create-stage)

## 6B. Alternative: create the stage with temporary credentials

Use this only when the sandbox supplies an access key, secret, **and session token**. Copy all three from the same credential set.

```sql
CREATE STAGE ORDERS_S3_STAGE
  URL = 's3://YOUR_UNIQUE_BUCKET/orders/'
  CREDENTIALS = (
    AWS_KEY_ID = 'REPLACE_WITH_TEMP_ACCESS_KEY_ID'
    AWS_SECRET_KEY = 'REPLACE_WITH_TEMP_SECRET_ACCESS_KEY'
    AWS_TOKEN = 'REPLACE_WITH_SESSION_TOKEN'
  )
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = FALSE)
  FILE_FORMAT = (FORMAT_NAME = ORDERS_CSV);

LIST @ORDERS_S3_STAGE;
```

Credentials expire according to the issuing session. Expiry requires fresh credentials and `ALTER STAGE`; recreating your orders table does not solve it. Creating a stage does not extend the sandbox lifetime.

[Credential parameters](https://docs.snowflake.com/en/sql-reference/sql/create-stage)

## 7. Compare stage listing with directory metadata

```sql
LIST @ORDERS_S3_STAGE;

ALTER STAGE ORDERS_S3_STAGE REFRESH;

SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED, FILE_URL
FROM DIRECTORY(@ORDERS_S3_STAGE)
ORDER BY RELATIVE_PATH;
```

`LIST` lists storage objects. `DIRECTORY` queries the stage's file metadata; refresh it after adding files in this manual-refresh lab. The directory should show `orders_01.csv`, not three order records. `FILE_URL` is a Snowflake file URL, not a public S3 link.

No SQS/SNS event setup is needed because directory automatic refresh is disabled.

[Directory-table management](https://docs.snowflake.com/en/user-guide/data-load-dirtables-manage)

## 8. Read CSV records directly from S3

```sql
SELECT
  T.$1::NUMBER AS ORDER_ID,
  T.$2::DATE AS ORDER_DATE,
  T.$3::VARCHAR AS CUSTOMER_NAME,
  T.$4::VARCHAR AS STATUS,
  T.$5::NUMBER(12,2) AS ORDER_TOTAL,
  METADATA$FILENAME AS SOURCE_FILE,
  METADATA$FILE_ROW_NUMBER AS SOURCE_ROW
FROM @ORDERS_S3_STAGE (FILE_FORMAT => 'ORDERS_CSV') T
ORDER BY ORDER_ID;
```

Expected: three orders, IDs 1001?1003. `$1` through `$5` are CSV fields by position. No table load has happened yet. This query verifies object reads as well as parsing; a successful `LIST` alone verifies neither.

[Query staged files](https://docs.snowflake.com/en/user-guide/querying-stage)

## 9. Load one native ORDERS table

```sql
CREATE TABLE IF NOT EXISTS ORDERS (
  ORDER_ID NUMBER,
  ORDER_DATE DATE,
  CUSTOMER_NAME VARCHAR,
  STATUS VARCHAR,
  ORDER_TOTAL NUMBER(12,2)
);

-- Validate the first file without loading it.
COPY INTO ORDERS
FROM @ORDERS_S3_STAGE
FILES = ('orders_01.csv')
FILE_FORMAT = (FORMAT_NAME = ORDERS_CSV)
VALIDATION_MODE = RETURN_ERRORS;

-- If validation reports no errors, load it.
COPY INTO ORDERS
FROM @ORDERS_S3_STAGE
FILES = ('orders_01.csv')
FILE_FORMAT = (FORMAT_NAME = ORDERS_CSV)
ON_ERROR = ABORT_STATEMENT;

SELECT * FROM ORDERS ORDER BY ORDER_ID;
SELECT COUNT(*) AS ORDER_COUNT, SUM(ORDER_TOTAL) AS TOTAL_AMOUNT
FROM ORDERS;
```

Expected on a fresh lab: **3 rows, total 445.75**. The native table stores a copy. It remains queryable after AWS credentials expire. Inspect each COPY result for loaded rows or errors.

Re-running COPY normally skips unchanged files already loaded into this table while its load metadata remains available. Keep `FORCE` off; it can reload rows. This is file-load tracking, not business-key deduplication.

[COPY INTO reference](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table)

## 10. Optional: query the same orders through an external table

This succeeds only with the **external S3 stage** above. A Snowflake-managed internal stage cannot be substituted. The external table is another representation of the same orders dataset, not a second business example.

```sql
CREATE EXTERNAL TABLE ExternalOrderTable (
  ORDER_ID NUMBER AS (VALUE:c1::NUMBER),
  ORDER_DATE DATE AS (VALUE:c2::DATE),
  CUSTOMER_NAME VARCHAR AS (VALUE:c3::VARCHAR),
  STATUS VARCHAR AS (VALUE:c4::VARCHAR),
  ORDER_TOTAL NUMBER(12,2) AS (VALUE:c5::NUMBER(12,2))
)
WITH LOCATION = @ORDERS_S3_STAGE
REFRESH_ON_CREATE = TRUE
AUTO_REFRESH = FALSE
FILE_FORMAT = (FORMAT_NAME = ORDERS_CSV);

SELECT ORDER_ID, ORDER_DATE, CUSTOMER_NAME, STATUS, ORDER_TOTAL
FROM ExternalOrderTable
ORDER BY ORDER_ID;
```

Expected: the same three orders. CSV records appear in `VALUE` as fields `c1`, `c2`, etc. External-table rows are read-only and depend on accessible S3 files. This object does not use the native `ORDERS` table.

[External-table syntax and restrictions](https://docs.snowflake.com/en/sql-reference/sql/create-external-table)

## 11. Add a second file: three different refresh actions

Upload the supplied `orders_02.csv` to the same S3 `orders/` prefix through the AWS console:

```csv
order_id,order_date,customer_name,status,order_total
1004,2026-09-12,Kiran,NEW,300.00
1005,2026-09-12,Divya,SHIPPED,49.50
```

```sql
-- See the newly uploaded object.
LIST @ORDERS_S3_STAGE;

-- Update file metadata in the directory table.
ALTER STAGE ORDERS_S3_STAGE REFRESH;
SELECT RELATIVE_PATH, SIZE
FROM DIRECTORY(@ORDERS_S3_STAGE)
ORDER BY RELATIVE_PATH;

-- Run only if you created the optional external table.
ALTER EXTERNAL TABLE ExternalOrderTable REFRESH;
SELECT COUNT(*) AS ORDER_COUNT, SUM(ORDER_TOTAL) AS TOTAL_AMOUNT
FROM ExternalOrderTable;

-- Load the second file into the native table separately.
COPY INTO ORDERS
FROM @ORDERS_S3_STAGE
FILES = ('orders_02.csv')
FILE_FORMAT = (FORMAT_NAME = ORDERS_CSV)
ON_ERROR = ABORT_STATEMENT;

SELECT COUNT(*) AS ORDER_COUNT, SUM(ORDER_TOTAL) AS TOTAL_AMOUNT
FROM ORDERS;
```

Expected: **2 CSV files** in the directory and **5 orders, total 795.25** in each table after its respective operation. Directory refresh does not refresh the external table or load the native table. External-table refresh does not load `ORDERS`.

[Directory refresh](https://docs.snowflake.com/en/user-guide/data-load-dirtables-manage), [external-table refresh](https://docs.snowflake.com/en/sql-reference/sql/alter-external-table).

## 12. Renew credentials without recreating the stage

If temporary credentials expire, obtain a fresh permitted set from the sandbox and run:

```sql
ALTER STAGE ORDERS_S3_STAGE SET CREDENTIALS = (
  AWS_KEY_ID = 'REPLACE_WITH_NEW_TEMP_ACCESS_KEY_ID'
  AWS_SECRET_KEY = 'REPLACE_WITH_NEW_TEMP_SECRET_ACCESS_KEY'
  AWS_TOKEN = 'REPLACE_WITH_NEW_SESSION_TOKEN'
);
LIST @ORDERS_S3_STAGE;
```

For rotation of an IAM-user key pair, use the same command with only `AWS_KEY_ID` and `AWS_SECRET_KEY`. These examples rotate credentials within the same authentication type. A restarted sandbox may also have a different bucket/account: confirm the URL and permissions first.

Avoid `CREATE OR REPLACE STAGE` for renewal because replacing a stage breaks its association with an existing external table. [ALTER STAGE](https://docs.snowflake.com/en/sql-reference/sql/alter-stage), [stage replacement behavior](https://docs.snowflake.com/en/sql-reference/sql/create-stage).

## 13. Troubleshooting and when a role becomes necessary

| Symptom | Check / next action |
|---|---|
| Invalid access key / signature error | Verify the ID and secret belong together; remove accidental whitespace |
| Invalid or expired token | Include `AWS_TOKEN` for temporary credentials; renew all three values together |
| AccessDenied on LIST | Check bucket URL, prefix, `s3:ListBucket`, and sandbox/bucket explicit denies |
| LIST works but row query fails | Check `s3:GetObject`, CSV settings, and any KMS decrypt permissions |
| Directory is empty | Enable directory metadata and run `ALTER STAGE ... REFRESH` |
| New CSV absent from external table | Run `ALTER EXTERNAL TABLE ... REFRESH` separately |
| External table rejects internal stage | Use an external S3 stage, or load an internal stage into a native table |
| COPY loads zero new rows | Inspect load results: the file may already have been loaded |
| IAM user/key creation denied | Use credentials already provided if permitted; otherwise ask the sandbox provider for a suitable lab |
| Snowflake requires storage integration | Ask the account administrator to provision an approved integration |

The access-key route cannot override sandbox permissions or an account rule requiring integrations. If that rule applies, role-based storage integration is needed and requires AWS trust configuration plus Snowflake setup. A role will not automatically solve an AWS explicit deny. [Role-based alternative](https://docs.snowflake.com/en/user-guide/data-load-s3-config-storage-integration)

If neither permitted keys nor an integration is available, use an internal Snowflake stage for upload, LIST, DIRECTORY, and COPY demonstrations. That fallback is not S3 integration and cannot back an external table. No role/trust-policy setup is required for the main lab.

## 14. Optional cleanup after class

Run only when finished with this lab's objects, with the same database and schema selected in Snowsight:

```sql
DROP EXTERNAL TABLE IF EXISTS ExternalOrderTable;
DROP TABLE IF EXISTS ORDERS;
DROP STAGE IF EXISTS ORDERS_S3_STAGE;
DROP FILE FORMAT IF EXISTS ORDERS_CSV;
```

Dropping an external stage does not delete its S3 files. In AWS, delete only the two lab CSVs and the lab bucket if you created it solely for this exercise. Deactivate/delete any IAM access key you created, then remove your lab IAM user and policy when no longer needed. Leave sandbox-managed identities unchanged.

The SQL is documented and reviewed, but has not been executed against your Snowflake trial or sandbox. The decisive checks are `LIST`, the staged-row query, and the expected counts above.